In [ ]:
"""
NOTEBOOK: USER SEGMENTATION
============================
Purpose: Cluster users into segments for targeted features
Output: User segments and profiles
"""

# 👥 User Segmentation Notebook

**Objective:** Group farmers and buyers into meaningful segments

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

class UserSegmenter:
    """
    PRODUCTION-READY USER SEGMENTATION
    Exports to: backend/app/ml/user_segmenter.py
    """
    
    def __init__(self, n_segments=4):
        self.n_segments = n_segments
        self.model = None
        self.scaler = StandardScaler()
        self.segment_profiles = {}
    
    def prepare_features(self, df):
        """Prepare user features for clustering"""
        
        # Aggregate user-level features
        user_features = df.groupby('user_id').agg({
            'transaction_id': 'count',
            'total_amount': 'mean',
            'quantity_kg': 'mean',
            'farmer_trust_score': 'mean',
            'is_completed': 'mean'
        }).reset_index()
        
        user_features.columns = ['user_id', 'txn_count', 'avg_amount', 
                                  'avg_quantity', 'trust_score', 'success_rate']
        
        # Additional features
        user_features['is_active'] = (user_features['txn_count'] > 5).astype(int)
        user_features['is_high_value'] = (user_features['avg_amount'] > 200).astype(int)
        
        features = user_features[['txn_count', 'avg_amount', 'trust_score', 
                                   'success_rate', 'is_active', 'is_high_value']]
        
        return features, user_features['user_id']
    
    def train(self, df):
        """Train segmentation model using KMeans"""
        print("🎯 Training user segmentation model...")
        
        X, user_ids = self.prepare_features(df)
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Apply PCA for visualization
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        
        # Train KMeans
        self.model = KMeans(n_clusters=self.n_segments, random_state=42, n_init=10)
        segments = self.model.fit_predict(X_scaled)
        
        # Create segment profiles
        X['segment'] = segments
        X['user_id'] = user_ids
        
        for seg in range(self.n_segments):
            segment_data = X[X['segment'] == seg]
            self.segment_profiles[seg] = {
                'size': len(segment_data),
                'avg_txn_count': segment_data['txn_count'].mean(),
                'avg_amount': segment_data['avg_amount'].mean(),
                'avg_trust': segment_data['trust_score'].mean(),
                'success_rate': segment_data['success_rate'].mean(),
                'name': self._name_segment(segment_data)
            }
        
        print(f"✅ Created {self.n_segments} user segments")
        
        for seg, profile in self.segment_profiles.items():
            print(f"\n   Segment {seg}: {profile['name']}")
            print(f"      Size: {profile['size']} users")
            print(f"      Avg Trust: {profile['avg_trust']:.0f}")
            print(f"      Success Rate: {profile['success_rate']:.1%}")
        
        return segments, X_pca
    
    def _name_segment(self, data):
        """Give meaningful names to segments"""
        if data['txn_count'].mean() > 20 and data['trust_score'].mean() > 80:
            return "Power Users"
        elif data['avg_amount'].mean() > 300:
            return "High Value Traders"
        elif data['success_rate'].mean() > 0.9:
            return "Reliable Traders"
        elif data['txn_count'].mean() < 3:
            return "New Users"
        else:
            return "Regular Users"
    
    def predict_segment(self, user_features):
        """Predict segment for a new user"""
        if self.model is None:
            raise ValueError("Model not trained yet")
        
        features_scaled = self.scaler.transform(user_features)
        segment = self.model.predict(features_scaled)[0]
        
        return {
            'segment_id': int(segment),
            'segment_name': self.segment_profiles[segment]['name'],
            'profile': self.segment_profiles[segment]
        }
    
    def save_model(self, version="v1"):
        """Save segmentation model"""
        import joblib
        import os
        
        os.makedirs("../../backend/ml_weights", exist_ok=True)
        
        model_path = f"../../backend/ml_weights/segmenter_{version}.pkl"
        joblib.dump(self.model, model_path)
        
        scaler_path = f"../../backend/ml_weights/segmenter_scaler_{version}.pkl"
        joblib.dump(self.scaler, scaler_path)
        
        print(f"💾 Segmentation model saved to: {model_path}")
        return model_path

print("\n✅ User segmentation loaded!")